In [ ]:
# =========================
# FULL CODE: LOAD DATASET FROM GOOGLE DRIVE + TRAIN MODEL
# =========================

# ---- 1. Mount Google Drive ----
from google.colab import drive
drive.mount('/content/drive')

# ---- 2. Set dataset path ----
data_dir = "/content/drive/MyDrive/Chatbot/PlantVillage/plantvillage dataset/color"

# ---- 3. Imports ----
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import os
import json

# ---- 4. Basic configs ----
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 25

# ---- 5. Data Generators ----
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    width_shift_range=0.2,
    height_shift_range=0.2,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# ---- 6. Load Data ----
train_data = train_datagen.flow_from_directory(
    data_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_data = val_datagen.flow_from_directory(
    data_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

# ---- 7. Save class indices ----
class_indices = train_data.class_indices
print("Class Mapping:", class_indices)

with open("class_indices.json", "w") as f:
    json.dump(class_indices, f)

# ---- 8. Load Pretrained MobileNet ----
base_model = tf.keras.applications.MobileNet(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False  # Freeze base model

# ---- 9. Build Model ----
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.2)(x)
outputs = keras.layers.Dense(len(class_indices), activation='softmax')(x)

model = keras.Model(inputs, outputs, name="PlantDisease_MobileNet")

# ---- 10. Compile ----
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ---- 11. Train ----
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS
)

# ---- 12. Plot Accuracy & Loss ----
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(len(acc))

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(epochs_range, acc, label='Train Accuracy')
plt.plot(epochs_range, val_acc, label='Val Accuracy')
plt.legend()
plt.title("Accuracy")

plt.subplot(1,2,2)
plt.plot(epochs_range, loss, label='Train Loss')
plt.plot(epochs_range, val_loss, label='Val Loss')
plt.legend()
plt.title("Loss")

plt.show()

# ---- 13. Save Model ----
model.save("/content/drive/MyDrive/plant_disease_model")

print("Model saved successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 43456 images belonging to 38 classes.
Found 10848 images belonging to 38 classes.
Class Mapping: {'Apple___Apple_scab': 0, 'Apple___Black_rot': 1, 'Apple___Cedar_apple_rust': 2, 'Apple___healthy': 3, 'Blueberry___healthy': 4, 'Cherry_(including_sour)___Powdery_mildew': 5, 'Cherry_(including_sour)___healthy': 6, 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 7, 'Corn_(maize)___Common_rust_': 8, 'Corn_(maize)___Northern_Leaf_Blight': 9, 'Corn_(maize)___healthy': 10, 'Grape___Black_rot': 11, 'Grape___Esca_(Black_Measles)': 12, 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)': 13, 'Grape___healthy': 14, 'Orange___Haunglongbing_(Citrus_greening)': 15, 'Peach___Bacterial_spot': 16, 'Peach___healthy': 17, 'Pepper,_bell___Bacterial_spot': 18, 'Pepper,_bell___healthy': 19, 'Potato___Early_blight': 20, 'Potato___Late_blight': 21, 'Potato___healthy': 22, 'Raspb

Model: "PlantDisease_MobileNet"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenet_1.00_224 (Functional) │ (None, 7, 7, 1024)     │     3,228,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1024)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 38)             │        38,950 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,267,814 (12.47 MB)

 Trainable params: 38,950 (152.15 KB)

 Non-trainable params: 3,228,864 (12.32 MB)

Epoch 1/25
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 14257s 10s/step - accuracy: 0.8508 - loss: 0.5257 - val_accuracy: 0.9383 - val_loss: 0.1952
Epoch 2/25
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 592s 436ms/step - accuracy: 0.9308 - loss: 0.2174 - val_accuracy: 0.9490 - val_loss: 0.1552
Epoch 3/25
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 591s 436ms/step - accuracy: 0.9395 - loss: 0.1856 - val_accuracy: 0.9522 - val_loss: 0.1418
Epoch 4/25
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 590s 434ms/step - accuracy: 0.9430 - loss: 0.1708 - val_accuracy: 0.9582 - val_loss: 0.1235
Epoch 5/25
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 593s 437ms/step - accuracy: 0.9462 - loss: 0.1622 - val_accuracy: 0.9644 - val_loss: 0.1095
Epoch 6/25
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 591s 435ms/step - accuracy: 0.9468 - loss: 0.1558 - val_accuracy: 0.9491 - val_loss: 0.1478
Epoch 7/25
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 575s 423ms/step - accuracy: 0.9494 - loss: 0.1499 - val_accuracy: 0.9628 - val_loss: 0.1089
Epoch 8/25
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 566s 417ms/step - ac

KeyboardInterrupt: 

In [ ]:
model.save("/content/drive/MyDrive/plant_disease_model_final.keras")